In [ ]:
import re
from html import escape


# ══════════════════════════════════════════════════════════════════════════
# Color Schemes
# ══════════════════════════════════════════════════════════════════════════

COLORS_DARK = {
    "keyword":     "#569CD6",
    "builtin":     "#DCDCAA",
    "type":        "#4EC9B0",
    "string":      "#CE9178",
    "number":      "#B5CEA8",
    "comment":     "#6A9955",
    "operator":    "#D4D4D4",
    "punctuation": "#D4D4D4",
    "parameter":   "#9CDCFE",
    "field":       "#9CDCFE",
    "default":     "#D4D4D4",
    "background":  "#1E1E1E",
}

COLORS_LIGHT = {
    "keyword":     "#0000FF",
    "builtin":     "#795E26",
    "type":        "#267F99",
    "string":      "#A31515",
    "number":      "#098658",
    "comment":     "#008000",
    "operator":    "#000000",
    "punctuation": "#000000",
    "parameter":   "#001080",
    "field":       "#001080",
    "default":     "#000000",
    "background":  "#FFFFFF",
}


# ══════════════════════════════════════════════════════════════════════════
# M-Code Keywords & Types
# ══════════════════════════════════════════════════════════════════════════

KEYWORDS = {
    "let", "in", "if", "then", "else", "each", "and", "or", "not",
    "true", "false", "null", "try", "otherwise", "error", "meta",
    "section", "shared", "as", "is", "type",
}

TYPES = {
    "text", "number", "logical", "date", "time", "datetime",
    "datetimezone", "duration", "record", "list", "table",
    "function", "binary", "none", "any", "anynonnull",
    "Int64.Type", "Int32.Type", "Percentage.Type",
    "Currency.Type", "Text.Type", "Logical.Type",
}


# ══════════════════════════════════════════════════════════════════════════
# Tokenizer
# ══════════════════════════════════════════════════════════════════════════

TOKEN_PATTERNS = [
    ("comment",     r'/\*[\s\S]*?\*/'),
    ("comment",     r'//[^\n]*'),
    ("string",      r'"(?:[^"\\]|\\.)*"'),
    ("field",       r'\[[^\]]*\]'),
    ("builtin",     r'[A-Z][a-zA-Z0-9]*\.[A-Za-z][A-Za-z0-9]*(?:\.[A-Za-z][A-Za-z0-9]*)?'),
    ("number",      r'\b\d+(?:\.\d+)?\b'),
    ("identifier",  r'[a-zA-Z_#][a-zA-Z0-9_]*'),
    ("operator",    r'<>|<=|>=|=>|->|\.\.\.|\.\.|[+\-*/&<>=@!?]'),
    ("punctuation", r'[(){}\[\],;]'),
    ("newline",     r'\n'),
    ("whitespace",  r'[ \t]+'),
    ("default",     r'.'),
]

TOKEN_REGEX = re.compile(
    '|'.join(f'(?P<{name}_{i}>{pattern})' for i, (name, pattern) in enumerate(TOKEN_PATTERNS)),
    re.MULTILINE
)


def _classify_token(token_type, token_value):
    """Determine the color category of a token."""
    if token_type == "identifier":
        lower = token_value.lower()
        if lower in KEYWORDS:
            return "keyword"
        if token_value in TYPES or lower in TYPES:
            return "type"
        return "default"
    return token_type


def _tokenize(m_code):
    """Split M-Code into a list of (type, value) tuples."""
    tokens = []
    for match in TOKEN_REGEX.finditer(m_code):
        for key, value in match.groupdict().items():
            if value is not None:
                token_type = key.rsplit('_', 1)[0]
                tokens.append((token_type, value))
                break
    return tokens


# ══════════════════════════════════════════════════════════════════════════
# M-Code Formatter (Auto-Indent)
# ══════════════════════════════════════════════════════════════════════════


def format_mcode(m_code, indent_str="    "):
    """
    Auto-format and indent Power Query M-Code.

    Rules:
      - 'let' starts a block, step assignments indented one level
      - 'in' closes the let block, result indented one level
      - Commas at step level start a new line
      - if/then/else get proper indentation
      - Parentheses and braces increase/decrease indent
      - Multi-line comments are preserved and indented

    Args:
        m_code:      M-Code as string
        indent_str:  Indentation string (default: 4 spaces)

    Returns:
        Formatted M-Code string.
    """
    tokens = _tokenize(m_code)

    # Remove all existing whitespace/newlines (we rebuild formatting)
    clean_tokens = []
    for ttype, tval in tokens:
        if ttype in ("whitespace", "newline"):
            # Keep a single space marker between non-whitespace tokens
            if clean_tokens and clean_tokens[-1] != ("_space", " "):
                clean_tokens.append(("_space", " "))
        else:
            clean_tokens.append((ttype, tval))

    # Remove trailing space
    if clean_tokens and clean_tokens[-1] == ("_space", " "):
        clean_tokens.pop()

    output = []
    indent = 0
    paren_depth = 0          # Track nested parentheses
    brace_depth = 0          # Track nested braces { }
    at_line_start = True
    in_let_block = False
    prev_keyword = ""

    def emit_newline():
        nonlocal at_line_start
        # Avoid double newlines
        stripped = ''.join(output).rstrip(' ')
        output.clear()
        output.append(stripped)
        output.append("\n")
        output.append(indent_str * indent)
        at_line_start = True

    def emit_token(val):
        nonlocal at_line_start
        output.append(val)
        at_line_start = False

    i = 0
    while i < len(clean_tokens):
        ttype, tval = clean_tokens[i]

        # Get keyword classification
        is_kw = (ttype == "identifier" and tval.lower() in KEYWORDS)
        kw = tval.lower() if is_kw else ""

        # ── let ───────────────────────────────────────────────────────
        if kw == "let":
            if not at_line_start:
                emit_newline()
            emit_token(tval)
            indent += 1
            emit_newline()
            in_let_block = True
            prev_keyword = "let"
            i += 1
            continue

        # ── in ────────────────────────────────────────────────────────
        if kw == "in" and paren_depth == 0:
            indent = max(indent - 1, 0)
            emit_newline()
            emit_token(tval)
            indent += 1
            emit_newline()
            in_let_block = False
            prev_keyword = "in"
            i += 1
            continue

        # ── if ────────────────────────────────────────────────────────
        if kw == "if":
            if prev_keyword == "else":
                # else if on same line
                if not (output and output[-1].endswith(" ")):
                    emit_token(" ")
                emit_token(tval)
            else:
                if not at_line_start:
                    emit_newline()
                emit_token(tval)
                indent += 1
            prev_keyword = "if"
            i += 1
            continue

        # ── then ──────────────────────────────────────────────────────
        if kw == "then":
            if not (output and output[-1].endswith(" ")):
                emit_token(" ")
            emit_token(tval)
            emit_token(" ")
            prev_keyword = "then"
            i += 1
            continue

        # ── else ──────────────────────────────────────────────────────
        if kw == "else":
            emit_newline()
            emit_token(tval)
            # Check if next non-space token is 'if' (else if)
            j = i + 1
            while j < len(clean_tokens) and clean_tokens[j][0] == "_space":
                j += 1
            if j < len(clean_tokens) and clean_tokens[j][0] == "identifier" and clean_tokens[j][1].lower() == "if":
                # else if – don't increase indent, 'if' handler will deal with it
                prev_keyword = "else"
            else:
                emit_token(" ")
                indent = max(indent - 1, 0)
                prev_keyword = "else"
            i += 1
            continue

        # ── each ──────────────────────────────────────────────────────
        if kw == "each":
            emit_token(tval)
            emit_token(" ")
            prev_keyword = "each"
            i += 1
            continue

        # ── Comma at step level ───────────────────────────────────────
        if ttype == "punctuation" and tval == ",":
            emit_token(tval)
            if paren_depth == 0 and brace_depth == 0 and in_let_block:
                # Top-level step separator → new line
                emit_newline()
            else:
                emit_token(" ")
            i += 1
            continue

        # ── Open paren ────────────────────────────────────────────────
        if ttype == "punctuation" and tval == "(":
            emit_token(tval)
            paren_depth += 1
            i += 1
            continue

        # ── Close paren ───────────────────────────────────────────────
        if ttype == "punctuation" and tval == ")":
            paren_depth = max(paren_depth - 1, 0)
            emit_token(tval)
            i += 1
            continue

        # ── Open brace ────────────────────────────────────────────────
        if ttype == "punctuation" and tval == "{":
            emit_token(tval)
            brace_depth += 1
            i += 1
            continue

        # ── Close brace ───────────────────────────────────────────────
        if ttype == "punctuation" and tval == "}":
            brace_depth = max(brace_depth - 1, 0)
            emit_token(tval)
            i += 1
            continue

        # ── Comments ──────────────────────────────────────────────────
        if ttype == "comment":
            if not at_line_start:
                emit_newline()
            # Handle multi-line comments: indent each line
            lines = tval.split("\n")
            for li, line in enumerate(lines):
                if li > 0:
                    output.append("\n")
                    output.append(indent_str * indent)
                output.append(line.strip() if li > 0 else line)
            emit_newline()
            i += 1
            continue

        # ── Spaces ────────────────────────────────────────────────────
        if ttype == "_space":
            if not at_line_start:
                # Don't add space if next token is operator (operator adds its own)
                if i + 1 < len(clean_tokens) and clean_tokens[i + 1][0] == "operator":
                    pass
                # Don't add space if output already ends with space
                elif output and output[-1].endswith(" "):
                    pass
                else:
                    emit_token(" ")
            i += 1
            continue

        # ── Operators ─────────────────────────────────────────────────
        if ttype == "operator":
            if not (output and output[-1].endswith(" ")):
                emit_token(" ")
            emit_token(tval)
            emit_token(" ")
            i += 1
            continue

        # ── Everything else ───────────────────────────────────────────
        emit_token(tval)
        if is_kw:
            prev_keyword = kw
        i += 1

    result = ''.join(output).strip()
    # Clean up multiple blank lines
    result = re.sub(r'\n{3,}', '\n\n', result)
    # Clean up trailing whitespace on lines
    result = re.sub(r' +\n', '\n', result)
    return result


# ══════════════════════════════════════════════════════════════════════════
# HTML Highlighting
# ══════════════════════════════════════════════════════════════════════════


def mcode_to_html(m_code, theme="dark", show_line_numbers=True,
                  font_size="13px", font_family="Consolas, 'Courier New', monospace",
                  colors=None, auto_format=True):
    """
    Convert Power Query M-Code to color-coded HTML.

    Args:
        m_code:             M-Code as string
        theme:              'dark' or 'light' (default: 'dark')
        show_line_numbers:  Show line numbers (default: True)
        font_size:          Font size (default: '13px')
        font_family:        Font family (default: Consolas)
        colors:             Optional custom color dict (overrides theme)
        auto_format:        Auto-indent the M-Code before highlighting (default: True)

    Returns:
        Full HTML string with wrapper div, suitable for notebook display.
    """
    if auto_format:
        m_code = format_mcode(m_code)

    base = COLORS_DARK if theme == "dark" else COLORS_LIGHT
    c = {**base, **(colors or {})}

    tokens = _tokenize(m_code)
    html_parts = []

    for token_type, token_value in tokens:
        if token_type in ("whitespace", "newline"):
            html_parts.append(token_value)
            continue
        category = _classify_token(token_type, token_value)
        color = c.get(category, c["default"])
        escaped = escape(token_value)
        if category == "keyword":
            html_parts.append(f'<span style="color:{color};font-weight:bold;">{escaped}</span>')
        else:
            html_parts.append(f'<span style="color:{color};">{escaped}</span>')

    highlighted = "".join(html_parts)

    if show_line_numbers:
        lines = highlighted.split("\n")
        pad = len(str(len(lines)))
        numbered = []
        for i, line in enumerate(lines, 1):
            num = f'<span style="color:#858585;user-select:none;">{str(i).rjust(pad)}  │ </span>'
            numbered.append(num + line)
        highlighted = "\n".join(numbered)

    return (
        f'<div style="background-color:{c["background"]};border-radius:6px;'
        f'padding:16px 20px;overflow-x:auto;font-family:{font_family};'
        f'font-size:{font_size};line-height:1.6;">'
        f'<pre style="margin:0;white-space:pre-wrap;word-wrap:break-word;">'
        f'{highlighted}</pre></div>'
    )


def mcode_to_ssrs_html(m_code, colors=None, auto_format=True):
    """
    Convert Power Query M-Code to HTML spans only (no wrapper).
    Suitable for SSRS / Paginated Report textboxes with Markup=HTML.
    Uses the light color scheme by default.

    Args:
        m_code:       M-Code as string
        colors:       Optional custom color dict (overrides light theme)
        auto_format:  Auto-indent before highlighting (default: True)

    Returns:
        HTML string with colored <span> tags and <br/> for line breaks.
    """
    if auto_format:
        m_code = format_mcode(m_code)

    c = {**COLORS_LIGHT, **(colors or {})}

    tokens = _tokenize(m_code)
    html_parts = []

    for token_type, token_value in tokens:
        if token_type in ("whitespace", "newline"):
            html_parts.append(token_value.replace("\n", "<br/>").replace(" ", "&nbsp;"))
            continue
        category = _classify_token(token_type, token_value)
        color = c.get(category, c["default"])
        escaped = escape(token_value)
        if category == "keyword":
            html_parts.append(f'<span style="color:{color};font-weight:bold;">{escaped}</span>')
        else:
            html_parts.append(f'<span style="color:{color};">{escaped}</span>')

    return "".join(html_parts)


def display_mcode(m_code, theme="dark", auto_format=True, **kwargs):
    """
    Render color-coded M-Code directly in a Fabric/Jupyter notebook.

    Args:
        m_code:       M-Code as string
        theme:        'dark' or 'light' (default: 'dark')
        auto_format:  Auto-indent before highlighting (default: True)
        **kwargs:     Additional arguments passed to mcode_to_html()
    """
    from IPython.display import HTML, display as ipy_display
    html = mcode_to_html(m_code, theme=theme, auto_format=auto_format, **kwargs)
    ipy_display(HTML(html))


print("✅ M-Code Highlighter loaded. Available functions:")
print("   format_mcode(m_code)                       → auto-indent M-Code")
print("   display_mcode(m_code, theme='dark')         → render in notebook")
print("   mcode_to_html(m_code, theme='dark')         → full HTML string")
print("   mcode_to_ssrs_html(m_code)                  → HTML spans for Paginated Reports")
print("")
print("   Set auto_format=False to skip auto-indentation.")